In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import optimize
from scipy import stats
import statsmodels.formula.api as smf 

# Also - set up plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
plt.style.use('fivethirtyeight')

### Binomial regression (also called logistic regression) - 
### Part 1 - changing two-alternative data into data suitable for linear regression tools

There is more detail about this topic in the textbook here: https://uob-ds.github.io/cfd2022/more-regression/logistic_regression.html

- Binomial (two-number) regression is used when we are predicting something that can only take two values
    - Dead/alive
    - Success/failure
    - Disease/no disease
    - Presence/Absence
<br>

We can't use normal linear regression techniques for this situation.  Why not?

#### Linear regression wants:
- A linear relationship
- Unbounded numbers
    - Continuous values
    - Can be negative or positive

### Why aren't  two-value situations like this?

- Only two possible values
- We are usually predicting the **_probability_** of the two options
    - e.g. The probability of disease vs no disease
    
### And so?
- If predicted values are probabilities...
    - Values are not linear (see below)
    - Values are bounded between zero and 1
        - no negative numbers
        - no numbers larger than one

### NOTE:  We are always talking about _predicted_ values here

- The observed values are what they are.  We can't mess with these
  -  Observed values are always 0 or 1
  - We are not trying to change anything about observed values
- For each observed value, however, we need to generate a _predicted_ value
  -  The predicted value is not a 0 or a 1 -- it is the _probability_ of zero or 1
 

#### Oops - that doesn't match what linear regression produces. Simple linear regression produces:
- Values on a line.
- Predicted values from an _**unlimited**_ set of numbers, not just 2 possibilities
  - i.e. Predicted values are not _bounded_
  - The values are not restricted to being between 0 and 1
 How do we match what we need to what linear regression produces?
 - We need predicted probabilities that change in a non-linear fashion (S-shaped -- see below)
 - We need predicted probabilities that are restricted to being between 0 and 1

### Why should predicted values be non-linear?

<img src=sigmoid_function.png width=400 align=right>

- The typical function that describes probabilities in a two-value situation is a "logistic function"
- This is an S-shaped curve that increases slowly at the beginning, faster in the middle and slowly again at the end
- A typical example is learning a new sport.  Think of training time on the x axis and ability on the y-axis.
- At the very beginning progress is slow.
- Once you start to get the hang of the basics you improve more rapidly
- To get to expert level, though, is much slower.
    - The final steps to reach a high level take a long time
- Our example below will be dunking a basketball and height
  - When people are sufficiently short, nobody can dunk a basketball
  - Once a certain height is reached a few people (good jumpers) can do it
  - When people are even taller, the change is rapid and many more people start to be able to dunk
  - When people are really tall (like most pros) everyone can dunk
 <br>
 
 #### This is not a linear function and it is bounded


### We only know about linear regression - what to do?


- Call Mick (https://www.youtube.com/watch?v=oqMl5CRoFdk)
    - Oh dear - A reference to what my children call "daddy music" -- sorry.  Couldn't resist.
    - "You don't always get what you want...you just might find, you get what you need"
    - A collection of 70-year old geriatrics (at that time of the video)
    - Compare to the younger version (https://www.youtube.com/watch?v=ruTMp4_sy1E&list=RDEf9QnZVpVd8&index=2)
    - Well, we all get older....

<img src=RollingStones.png align=right width=500>

#### Don't have the tools for two choices directly?
- Get what you need:
    - Make the problem into a linear regression problem
    - Convert probabilities into another value that is:
      - linear
      - unbounded continuous
    - Use linear techniques to predict your converted value
    - Reconvert the predicted value back to a probability value
<br>

## How?

### First step - Convert from 0-1 to 0-unbounded

- Note that any probability in a two-choice situation has its opposite
    - The probability of success determines the probability of failure
    - The two options must add up to 1
    - If the probability of success is 80%, the probability of failure must be 20%
 <br>
 
 We can use this and turn any probability in an **_odds ratio_**
 
- An odds ratio asks: How much more likely is success than failure?
- 80% / 20% means that success is 4 times more likely than failure
- This fraction can take on any number from 0 to infinity
<br>
  
Odds ration equation: 
 
$$\frac{p(success)}{p(failure)}$$
 
 - or, since 
 
 $$p(failure) = 1 - p(success)$$
 
- odds ratio equation: 
 
$$\frac{p(success)}{(1 - p(success))}$$

Notice that, as we've said, an odds ratio is an unbounded number between 0 and positive infinity

#### Now we are half way there, but only halfway...
#### There is still a bound at 0

### Let's review where we are up to now

The possible outcomes are:
- success is more likely than failure
- failure is more likely than success
- even odds
<br>

- If even odds

$$p(success) = p(failure)$$  
$$\frac{p(success)}{p(failure)} = 1$$  

- If success more likely
$$\frac{p(success) = big\ number}{p(failure) = small\ number}$$  
    - Result > 1
    - range: 1 to infinity
    
- If failure more likely
$$\frac{p(success) = small\ number}{p(failure) = big\ number}$$    
    - Result between 1 and 0
    
    
That hardly seems fair.  
- When success is more likely we've got all the possible numbers > 1
- When failure is more likely our numbers are smashed between 0 and 1

Plus we are still bounded on the lower side by zero.

### Step 2 - Solution:  Take the log

You may remember that:

$$10^1 = 10$$
$$log_{10}{10} = 1$$
<br>
  
$$10^2 = 100$$
$$log_{10}(100) = 2$$  
<br>
  
$$10^3 = 1000$$
$$log_{10}(1000) = 3$$
<br>
  
General rule:
$$ x^y = z$$
$$ log_{x}(z) = y$$

You may also remember that
$$0.1 = 10^{-1}$$
$$0.01 = 10^{-2}$$
$$0.001 = 10^{-3}$$

So after taking the log, the numbers between 0 and 1 are converted to negative numbers between 0 and negative infinity.  Now we have numbers between negative infinity and positive infinity, with even odds at the zero point.

Our normal linear regression techinques can predict a log(odds_ratio) and then we can convert that to a probability to see how closely the probability matches the actual values.

## Summary

If we start from predicted probabilities:

p(failure) = 1 - p(success) 

$$odds\ ratio = \frac{p(success)}{(1 - p(success))}$$

$$log\ odds = log(\frac{p(success)}{p(failure)})$$

#### This converts probabilities of success/failure that are 0-1 to continuous numbers from -infinity to +infinity
#### We can also start by predicting log odds and go backward to get probability values when we want to compare to observed values (this is what the regression does)


#### You might see taking step 1 and step 2 together labelled -- "taking the log odds"


All righty then!  Now I can use my usual regression tools

#### Preparing for an analysis

- What this means for an analysis:
  - We will need a function that converts probabilities into log odds.
  - We will need a function that converts log odds into probabilities.

In [ ]:
# write a function to convert probabilities into log odds

In [ ]:
# write a function to convert log odds into probabilities